# Cruise Control Demo

An interactive PID cruise control simulation. Use the sliders to tune the controller, change the vehicle, and throw disturbances (hills) at it.

**Manufacturing bridge:** This is the same problem as controlling conveyor belt speed under varying product load, spindle speed during material engagement, or pump speed against changing system resistance.

**Vehicle model:**
```
m * dv/dt = F_engine - F_drag - F_rolling - F_hill
```
- `F_engine = throttle * engine_force_max` (throttle clipped to 0–1)
- `F_drag = 0.5 * Cd * A * rho * v^2` (aerodynamic drag)
- `F_rolling = Cr * m * g` (tire rolling resistance)
- `F_hill = m * g * sin(grade)` (gravity on a slope)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

import cruise_control_helpers as cc

print("Imports OK")

---
## Section 1 — Baseline Simulation (Run First)

Run this section to see the default cruise control behavior before interacting with widgets.

In [ ]:
# ---- Editable baseline parameters ----
TARGET_SPEED_KMH = 80.0   # Target cruise speed [km/h]
SCENARIO = "hill_profile"  # Options: flat, uphill, downhill, hill_profile, speed_change

Kp = 0.05
Ki = 0.01
Kd = 0.005
# --------------------------------------

In [ ]:
params = cc.default_params()
params.update(Kp=Kp, Ki=Ki, Kd=Kd)

t = np.arange(0, params["t_end"], params["dt"])
target, grade = cc.build_scenario(t, scenario=SCENARIO, target_speed_kmh=TARGET_SPEED_KMH)
result = cc.simulate(params, target, grade)

fig = cc.build_figure(result)
fig.update_layout(title_text=f"Baseline: {SCENARIO} scenario, target={TARGET_SPEED_KMH} km/h")
fig.show()

---
## Section 2 — Interactive Tuning

Move the sliders below to see how each parameter affects cruise control performance.

**Things to try:**
- Increase `Kp` and watch the speed respond faster (but overshoot more).
- Add `Ki` to eliminate the steady-state gap on uphills.
- Increase `Kd` to dampen overshoot.
- Switch to **hill_profile** and watch the controller fight the grade changes.
- Increase vehicle mass — notice how the same gains struggle with a heavier load.

In [ ]:
# Interactive widget section - requires a live Jupyter session.
# Falls back to a static plot when run headless (e.g., nbconvert).

try:
    fig_widget = go.FigureWidget(
        make_subplots(
            rows=3, cols=1, shared_xaxes=True,
            subplot_titles=("Vehicle Speed vs Target", "Throttle (Control Effort)", "Road Grade (Disturbance)"),
            vertical_spacing=0.08,
        )
    )

    # Pre-add empty traces
    fig_widget.add_trace(go.Scatter(x=[], y=[], mode="lines", name="Target speed",
                                    line=dict(color="black", dash="dash", width=1.5)), row=1, col=1)
    fig_widget.add_trace(go.Scatter(x=[], y=[], mode="lines", name="Vehicle speed",
                                    line=dict(color="#636EFA", width=2)), row=1, col=1)
    fig_widget.add_trace(go.Scatter(x=[], y=[], mode="lines", name="Throttle",
                                    line=dict(color="#EF553B", width=1.5)), row=2, col=1)
    fig_widget.add_trace(go.Scatter(x=[], y=[], mode="lines", name="Road grade",
                                    line=dict(color="#00CC96", width=1.5),
                                    fill="tozeroy", fillcolor="rgba(0,204,150,0.15)"), row=3, col=1)

    fig_widget.update_yaxes(title_text="Speed [km/h]", row=1, col=1)
    fig_widget.update_yaxes(title_text="Throttle [%]", range=[-5, 105], row=2, col=1)
    fig_widget.update_yaxes(title_text="Grade [%]", row=3, col=1)
    fig_widget.update_xaxes(title_text="Time [s]", row=3, col=1)
    fig_widget.update_layout(
        template="plotly_white", height=700,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        margin=dict(t=60, b=40),
    )

    # Sliders
    slider_style = dict(style={"description_width": "100px"}, layout=widgets.Layout(width="420px"))

    w_Kp = widgets.FloatSlider(value=0.05, min=0.0, max=0.30, step=0.005, description="Kp", **slider_style)
    w_Ki = widgets.FloatSlider(value=0.01, min=0.0, max=0.10, step=0.002, description="Ki", **slider_style)
    w_Kd = widgets.FloatSlider(value=0.005, min=0.0, max=0.10, step=0.002, description="Kd", **slider_style)
    w_mass = widgets.FloatSlider(value=1200, min=600, max=3000, step=100, description="Mass [kg]", **slider_style)
    w_Cd = widgets.FloatSlider(value=0.30, min=0.10, max=0.80, step=0.05, description="Drag Cd", **slider_style)
    w_target = widgets.FloatSlider(value=80, min=30, max=140, step=5, description="Target [km/h]", **slider_style)
    w_scenario = widgets.Dropdown(
        options=["flat", "uphill", "downhill", "hill_profile", "speed_change"],
        value="hill_profile", description="Scenario",
        style={"description_width": "100px"}, layout=widgets.Layout(width="420px"),
    )

    def update_sim(change=None):
        params = cc.default_params()
        params.update(
            Kp=w_Kp.value, Ki=w_Ki.value, Kd=w_Kd.value,
            m=w_mass.value, Cd=w_Cd.value,
        )
        t = np.arange(0, params["t_end"], params["dt"])
        target, grade = cc.build_scenario(t, scenario=w_scenario.value, target_speed_kmh=w_target.value)
        result = cc.simulate(params, target, grade)

        with fig_widget.batch_update():
            fig_widget.data[0].x = result["t"]
            fig_widget.data[0].y = result["target"] * 3.6
            fig_widget.data[1].x = result["t"]
            fig_widget.data[1].y = result["v"] * 3.6
            fig_widget.data[2].x = result["t"]
            fig_widget.data[2].y = result["throttle"] * 100
            fig_widget.data[3].x = result["t"]
            fig_widget.data[3].y = result["grade"] * 100

    for w in [w_Kp, w_Ki, w_Kd, w_mass, w_Cd, w_target, w_scenario]:
        w.observe(update_sim, names="value")

    controls_box = widgets.VBox([
        widgets.HTML("<b>PID Gains</b>"),
        w_Kp, w_Ki, w_Kd,
        widgets.HTML("<b>Vehicle / Scenario</b>"),
        w_mass, w_Cd, w_target, w_scenario,
    ])

    update_sim()
    display(widgets.HBox([controls_box, fig_widget]))

except (ImportError, Exception) as exc:
    # Fallback for headless or missing-widget environments
    print(f"(Interactive widgets unavailable: {exc})")
    print("Showing static plot with default parameters instead.")
    params = cc.default_params()
    t = np.arange(0, params["t_end"], params["dt"])
    target, grade = cc.build_scenario(t, scenario="hill_profile", target_speed_kmh=80.0)
    result = cc.simulate(params, target, grade)
    fig = cc.build_figure(result)
    fig.update_layout(title_text="Interactive Tuning (static fallback)")
    fig.show()

---
## Section 3 — Tuning Challenge

**Scenario:** `hill_profile`, target = 100 km/h, mass = 2000 kg

**Goal:** Find PID gains that achieve ALL of the following:
- Reach target speed within **20 seconds**
- Overshoot less than **5 km/h** above the target
- Recover from the hill within **10 seconds** of grade change
- Throttle never saturates for more than **5 seconds straight**

Enter your gains below and run the cell to check.

In [ ]:
# ---- YOUR GAINS HERE ----
challenge_Kp = 0.05
challenge_Ki = 0.01
challenge_Kd = 0.005
# -------------------------

In [ ]:
ch_params = cc.default_params()
ch_params.update(Kp=challenge_Kp, Ki=challenge_Ki, Kd=challenge_Kd, m=2000.0)

ch_t = np.arange(0, ch_params["t_end"], ch_params["dt"])
ch_target, ch_grade = cc.build_scenario(ch_t, scenario="hill_profile", target_speed_kmh=100.0)
ch_result = cc.simulate(ch_params, ch_target, ch_grade)

# Evaluate criteria
v_kmh = ch_result["v"] * 3.6
tgt_kmh = ch_result["target"] * 3.6
t = ch_result["t"]

# 1. Time to reach within 2 km/h of target
within_band = np.abs(v_kmh - tgt_kmh) < 2.0
reach_idx = np.where(within_band)[0]
reach_time = t[reach_idx[0]] if len(reach_idx) > 0 else None

# 2. Max overshoot
overshoot_kmh = np.max(v_kmh - tgt_kmh)

# 3. Max consecutive saturation time
saturated = ch_result["throttle"] >= 0.99
max_sat_run = 0
current_run = 0
for s in saturated:
    if s:
        current_run += 1
        max_sat_run = max(max_sat_run, current_run)
    else:
        current_run = 0
max_sat_seconds = max_sat_run * ch_params["dt"]

print("=" * 50)
print("  TUNING CHALLENGE RESULTS")
print("=" * 50)
print(f"  Reach target within 20s : {'PASS' if reach_time and reach_time < 20 else 'FAIL'}  (reached at {reach_time:.1f}s)" if reach_time else "  Reach target: FAIL (never reached)")
print(f"  Overshoot < 5 km/h      : {'PASS' if overshoot_kmh < 5 else 'FAIL'}  (max overshoot: {overshoot_kmh:.1f} km/h)")
print(f"  Sat. < 5s continuous     : {'PASS' if max_sat_seconds < 5 else 'FAIL'}  (longest sat: {max_sat_seconds:.1f}s)")
print("=" * 50)

fig = cc.build_figure(ch_result)
fig.update_layout(title_text="Tuning Challenge: 2000 kg vehicle, hill_profile, 100 km/h target")
fig.show()

---
## Bonus — P vs PI vs PID Comparison

See why each term matters by running the same scenario with P-only, PI, and full PID.

In [ ]:
comp_params = cc.default_params()
comp_params["m"] = 1500.0
comp_t = np.arange(0, comp_params["t_end"], comp_params["dt"])
comp_target, comp_grade = cc.build_scenario(comp_t, scenario="uphill", target_speed_kmh=90.0)

cases = {
    "P only  (Kp=0.08)":            dict(Kp=0.08, Ki=0.0,  Kd=0.0),
    "PI      (Kp=0.08, Ki=0.02)":   dict(Kp=0.08, Ki=0.02, Kd=0.0),
    "PID     (Kp=0.08, Ki=0.02, Kd=0.01)": dict(Kp=0.08, Ki=0.02, Kd=0.01),
}

colors = ["#636EFA", "#EF553B", "#00CC96"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Speed Comparison", "Throttle Comparison"),
                    vertical_spacing=0.10)

for (label, gains), color in zip(cases.items(), colors):
    p = {**comp_params, **gains}
    res = cc.simulate(p, comp_target, comp_grade)
    fig.add_trace(go.Scatter(x=res["t"], y=res["v"]*3.6, mode="lines",
                             name=label, line=dict(color=color, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=res["t"], y=res["throttle"]*100, mode="lines",
                             name=label, line=dict(color=color, width=1.5),
                             showlegend=False), row=2, col=1)

fig.add_trace(go.Scatter(x=comp_t, y=comp_target*3.6, mode="lines",
                         name="Target", line=dict(color="black", dash="dash", width=1.5)), row=1, col=1)

fig.update_yaxes(title_text="Speed [km/h]", row=1, col=1)
fig.update_yaxes(title_text="Throttle [%]", range=[-5, 105], row=2, col=1)
fig.update_xaxes(title_text="Time [s]", row=2, col=1)
fig.update_layout(template="plotly_white", height=550,
                  title_text="P vs PI vs PID on Uphill Scenario")
fig.show()

---

**Key takeaways:**
- P-only control leaves a speed gap on hills (steady-state error).
- Adding I eliminates the gap but can cause overshoot.
- Adding D dampens the overshoot and smooths the response.
- Actuator saturation (throttle 0–100%) is a real constraint in every physical system.

The same tuning tradeoffs apply to any speed/rate regulation problem in manufacturing.